# Keithley 2460 buffered acquisition and host-exception cleanup

Prepared next-step test for the nominal 100 Ω resistor. The notebook is safe to run offline through the dry-run section. The final hardware cell is deliberately latched off.

The hardware phase will:

1. Require SCPI mode, front terminals, and output OFF.
2. Create a uniquely named 32-reading user buffer in fill-once mode.
3. Source +1 mA with a 0.2 V limit and collect 20 readings at 1 NPLC.
4. Switch output OFF before bulk retrieval.
5. Retrieve source readback, four-wire voltage, relative time, source status, and measurement status.
6. Briefly enable the same bounded output and deliberately raise a host-side exception; `finally` must restore OFF and 0 A.
7. Delete only the uniquely named temporary buffer, close the session, and independently audit OFF/0 A in a new session.

This does **not** test process death, Ethernet loss, or instrument-local watchdog behavior. Do not run it with a battery or unknown DUT. If communication fails while output might be on, use the physical OUTPUT control immediately. Commands were reviewed against the official Model 2460 Reference Manual Rev. C, sections 6 and 12.

In [ ]:
from __future__ import annotations

import json
import math
import statistics
from datetime import UTC, datetime
from itertools import pairwise
from types import SimpleNamespace
from uuid import uuid4

from ophyd_electrochemistry.keithley.k2460.transport import (
    PyVisaKeithley2460Transport,
    VisaTransportConfig,
)

RESOURCE = "TCPIP0::169.254.77.125::5025::SOCKET"
CONFIRM_FRONT_4WIRE_100_OHM_AND_OUTPUT_OFF = False
BUFFER_CAPACITY = 32
SAMPLE_COUNT = 20
SOURCE_CURRENT_A = 1e-3
VOLTAGE_LIMIT_V = 0.2


class IntentionalHostTestError(RuntimeError):
    pass


def finite_number(reply: str, label: str) -> float:
    try:
        value = float(reply.strip())
    except ValueError as exc:
        raise RuntimeError(f"{label}: nonnumeric reply {reply!r}") from exc
    if not math.isfinite(value):
        raise RuntimeError(f"{label}: nonfinite reply {reply!r}")
    return value


def require_token(actual: str, allowed: tuple[str, ...], label: str) -> None:
    if actual not in allowed:
        raise RuntimeError(f"{label}: expected one of {allowed}, received {actual!r}")


def require_error_count_unchanged(instrument, expected: int, label: str) -> None:
    count = int(instrument.query(":SYSTem:ERRor:COUNt?"))
    if count != expected:
        raise RuntimeError(
            f"{label}: instrument error count changed from {expected} to {count}; "
            "inspect the front-panel event log without clearing it automatically"
        )


def parse_buffer_data(raw: str, expected_count: int) -> list[dict[str, float | int]]:
    fields = [field.strip() for field in raw.replace(";", ",").split(",") if field.strip()]
    fields_per_reading = 5
    if len(fields) != expected_count * fields_per_reading:
        raise RuntimeError(
            f"expected {expected_count * fields_per_reading} buffer fields, got {len(fields)}"
        )
    records = []
    for offset in range(0, len(fields), fields_per_reading):
        records.append(
            {
                "source_current_a": finite_number(fields[offset], "buffer source"),
                "measured_voltage_v": finite_number(fields[offset + 1], "buffer reading"),
                "relative_time_s": finite_number(fields[offset + 2], "buffer time"),
                "source_status": int(fields[offset + 3]),
                "measurement_status": int(fields[offset + 4]),
            }
        )
    return records

In [ ]:
def run_buffer_and_exception_test(instrument, *, buffer_name: str) -> dict[str, object]:
    report: dict[str, object] = {
        "schema": "k2460-buffer-cleanup-v1",
        "observed_at_utc": datetime.now(UTC).isoformat(),
        "buffer_name": buffer_name,
        "buffer_capacity": BUFFER_CAPACITY,
        "sample_count": SAMPLE_COUNT,
        "source_current_a": SOURCE_CURRENT_A,
        "voltage_limit_v": VOLTAGE_LIMIT_V,
    }
    buffer_created = False
    cleanup_error = None
    baseline_error_count = None

    try:
        instrument.connect()
        report["identity"] = instrument.identity.raw
        require_token(instrument.query(":OUTPut:STATe?"), ("0", "OFF"), "preflight output")
        require_token(instrument.query(":ROUTe:TERMinals?"), ("FRON", "FRONT"), "terminals")
        baseline_error_count = int(instrument.query(":SYSTem:ERRor:COUNt?"))
        report["preexisting_error_count"] = baseline_error_count

        for command in (
            ":FORMat:DATA ASCii",
            ":FORMat:ASCii:PRECision 16",
            ":SOURce:FUNCtion CURRent",
            ":SOURce:CURRent 0",
            ":SOURce:CURRent:RANGe 0.001",
            ":SOURce:CURRent:VLIMit 0.2",
            ":SOURce:CURRent:READ:BACK ON",
            ':SENSe:FUNCtion "VOLTage"',
            ":SENSe:VOLTage:RSENse ON",
            ":SENSe:VOLTage:RANGe 0.2",
            ":SENSe:VOLTage:NPLCycles 1",
            ":SENSe:VOLTage:AVERage OFF",
            ":SENSe:VOLTage:RELative:STATe OFF",
            ":SENSe:COUNt 1",
        ):
            instrument.write(command)

        require_token(instrument.query(":OUTPut:STATe?"), ("0", "OFF"), "configured output")
        require_token(instrument.query(":SOURce:FUNCtion?"), ("CURR", "CURRENT"), "source function")
        require_token(instrument.query(":SENSe:VOLTage:RSENse?"), ("1", "ON"), "remote sense")
        if (
            abs(
                finite_number(instrument.query(":SOURce:CURRent:VLIMit?"), "voltage limit")
                - VOLTAGE_LIMIT_V
            )
            > 1e-9
        ):
            raise RuntimeError("voltage limit verification failed")
        require_error_count_unchanged(instrument, baseline_error_count, "configuration")

        instrument.write(f':TRACe:MAKE "{buffer_name}", {BUFFER_CAPACITY}, STANdard')
        buffer_created = True
        if int(instrument.query(f':TRACe:POINts? "{buffer_name}"')) != BUFFER_CAPACITY:
            raise RuntimeError("buffer capacity verification failed")
        require_token(
            instrument.query(f':TRACe:FILL:MODE? "{buffer_name}"'),
            ("ONCE",),
            "buffer fill mode",
        )
        if int(instrument.query(f':TRACe:ACTual? "{buffer_name}"')) != 0:
            raise RuntimeError("new buffer is not empty")
        require_error_count_unchanged(instrument, baseline_error_count, "buffer creation")

        instrument.write(f":SENSe:COUNt {SAMPLE_COUNT}")
        instrument.write(f":SOURce:CURRent {SOURCE_CURRENT_A:.12g}")
        require_token(instrument.query(":OUTPut:STATe?"), ("0", "OFF"), "pre-acquisition output")
        try:
            instrument.write(":OUTPut:STATe ON")
            require_token(instrument.query(":OUTPut:STATe?"), ("1", "ON"), "acquisition output")
            report["last_reading_raw"] = instrument.query(
                f':READ? "{buffer_name}", SOURce, READing, RELative'
            )
        finally:
            instrument.write(":OUTPut:STATe OFF")
            require_token(
                instrument.query(":OUTPut:STATe?"), ("0", "OFF"), "post-acquisition output"
            )

        actual = int(instrument.query(f':TRACe:ACTual? "{buffer_name}"'))
        start = int(instrument.query(f':TRACe:ACTual:STARt? "{buffer_name}"'))
        end = int(instrument.query(f':TRACe:ACTual:END? "{buffer_name}"'))
        if (actual, start, end) != (SAMPLE_COUNT, 1, SAMPLE_COUNT):
            raise RuntimeError(f"unexpected buffer extent: {(actual, start, end)}")
        raw_data = instrument.query(
            f':TRACe:DATA? 1, {SAMPLE_COUNT}, "{buffer_name}", '
            "SOURce, READing, RELative, SOURSTATus, STATus"
        )
        records = parse_buffer_data(raw_data, SAMPLE_COUNT)
        times = [float(record["relative_time_s"]) for record in records]
        currents = [float(record["source_current_a"]) for record in records]
        voltages = [float(record["measured_voltage_v"]) for record in records]
        if any(later < earlier for earlier, later in pairwise(times)):
            raise RuntimeError("relative timestamps are not monotonic")
        if any(abs(current - SOURCE_CURRENT_A) > 5e-6 for current in currents):
            raise RuntimeError("source readback outside broad plausibility bound")
        resistance_values = [
            voltage / current for current, voltage in zip(currents, voltages, strict=True)
        ]
        if any(not 80 <= resistance <= 120 for resistance in resistance_values):
            raise RuntimeError("buffered resistance outside broad plausibility bound")
        if any(int(record["measurement_status"]) & 0x1 for record in records):
            raise RuntimeError("questionable measurement status in buffer")
        if any(not int(record["measurement_status"]) & 0x8 for record in records):
            raise RuntimeError("buffer record does not identify front terminals")
        require_token(
            instrument.query(":SOURce:CURRent:VLIMit:TRIPped?"),
            ("0",),
            "voltage-limit state",
        )
        require_error_count_unchanged(
            instrument, baseline_error_count, "buffer acquisition and retrieval"
        )
        report["records"] = records
        report["summary"] = {
            "mean_current_a": statistics.fmean(currents),
            "mean_voltage_v": statistics.fmean(voltages),
            "mean_resistance_ohm": statistics.fmean(resistance_values),
            "elapsed_s": times[-1] - times[0],
        }

        instrument.write(":SENSe:COUNt 1")
        instrument.write(":SOURce:CURRent 0")
        try:
            instrument.write(f":SOURce:CURRent {SOURCE_CURRENT_A:.12g}")
            instrument.write(":OUTPut:STATe ON")
            require_token(instrument.query(":OUTPut:STATe?"), ("1", "ON"), "exception-test output")
            raise IntentionalHostTestError("deliberate host-side test exception")
        except IntentionalHostTestError:
            report["intentional_exception_observed"] = True
        finally:
            instrument.write(":OUTPut:STATe OFF")
            require_token(
                instrument.query(":OUTPut:STATe?"), ("0", "OFF"), "exception cleanup output"
            )
            instrument.write(":SOURce:CURRent 0")
            report["exception_cleanup_output_off"] = True
            report["exception_cleanup_current_a"] = finite_number(
                instrument.query(":SOURce:CURRent?"), "exception cleanup current"
            )
    finally:
        if instrument.connected:
            try:
                instrument.write(":OUTPut:STATe OFF")
                require_token(instrument.query(":OUTPut:STATe?"), ("0", "OFF"), "final output")
                instrument.write(":SOURce:CURRent 0")
                instrument.write(":SENSe:COUNt 1")
                report["final_output_enabled"] = False
                report["final_programmed_current_a"] = finite_number(
                    instrument.query(":SOURce:CURRent?"), "final current"
                )
                if buffer_created:
                    instrument.write(f':TRACe:DELete "{buffer_name}"')
                    if baseline_error_count is None:
                        raise RuntimeError("missing baseline instrument error count")
                    require_error_count_unchanged(
                        instrument, baseline_error_count, "temporary-buffer deletion"
                    )
                    report["temporary_buffer_deleted"] = True
            except BaseException as exc:
                cleanup_error = exc
        try:
            instrument.close()
        except BaseException as exc:
            cleanup_error = cleanup_error or exc
        if cleanup_error is not None:
            raise RuntimeError(
                f"cleanup failed; inspect the front panel: {cleanup_error}"
            ) from cleanup_error

    return report

## Offline control-flow dry run

This fake implements only the reviewed replies needed by this notebook. It proves command ordering, parsing, the deliberate-exception path, buffer deletion, and final OFF/0 A cleanup; it is not firmware evidence.

In [ ]:
class OfflineFakeInstrument:
    def __init__(self) -> None:
        self.connected = False
        self.identity = SimpleNamespace(raw="KEITHLEY INSTRUMENTS,MODEL 2460,FAKE,0")
        self.output = False
        self.current_a = 0.0
        self.count = 1
        self.buffer_name = None
        self.buffer_deleted = False
        self.commands = []
        self.records = [
            (0.001, 0.1025 + index * 1e-7, index * 0.025, 0, 8) for index in range(SAMPLE_COUNT)
        ]
        self.acquired = False

    def connect(self) -> None:
        self.connected = True

    def close(self) -> None:
        self.connected = False

    def write(self, command: str) -> None:
        self.commands.append(command)
        if command == ":OUTPut:STATe ON":
            self.output = True
        elif command == ":OUTPut:STATe OFF":
            self.output = False
        elif command.startswith(":SOURce:CURRent "):
            self.current_a = float(command.rsplit(" ", 1)[1])
        elif command.startswith(":SENSe:COUNt "):
            self.count = int(command.rsplit(" ", 1)[1])
        elif command.startswith(":TRACe:MAKE "):
            self.buffer_name = command.split('"')[1]
        elif command.startswith(":TRACe:DELete "):
            self.buffer_deleted = True
            self.buffer_name = None

    def query(self, command: str) -> str:
        self.commands.append(command)
        if command == ":OUTPut:STATe?":
            return "1" if self.output else "0"
        if command == ":ROUTe:TERMinals?":
            return "FRON"
        if command == ":SOURce:FUNCtion?":
            return "CURR"
        if command == ":SENSe:VOLTage:RSENse?":
            return "1"
        if command == ":SOURce:CURRent:VLIMit?":
            return "0.2"
        if command == ":SOURce:CURRent:VLIMit:TRIPped?":
            return "0"
        if command == ":SYSTem:ERRor:COUNt?":
            return "0"
        if command == ":SOURce:CURRent?":
            return str(self.current_a)
        if command.startswith(":TRACe:POINts? "):
            return str(BUFFER_CAPACITY)
        if command.startswith(":TRACe:FILL:MODE? "):
            return "ONCE"
        if command.startswith(":TRACe:ACTual:STARt? "):
            return "1"
        if command.startswith(":TRACe:ACTual:END? "):
            return str(SAMPLE_COUNT)
        if command.startswith(":TRACe:ACTual? "):
            return str(SAMPLE_COUNT if self.acquired else 0)
        if command.startswith(":READ? "):
            self.acquired = True
            source, reading, relative, _, _ = self.records[-1]
            return f"{source},{reading},{relative}"
        if command.startswith(":TRACe:DATA? "):
            return ",".join(str(value) for record in self.records for value in record)
        raise RuntimeError(f"offline fake has no reply for {command!r}")


fake = OfflineFakeInstrument()
dry_run = run_buffer_and_exception_test(fake, buffer_name="oe_m3_fake")
assert fake.output is False
assert fake.current_a == 0.0
assert fake.buffer_deleted is True
assert dry_run["intentional_exception_observed"] is True
assert dry_run["exception_cleanup_output_off"] is True
assert dry_run["temporary_buffer_deleted"] is True
print("PASS: offline buffer, exception, and cleanup control flow")

## Hardware boundary — stop here while disconnected

Do not run the next cell until Ethernet and the four-wire 100 Ω resistor are reconnected, the front terminals are selected, the physical OUTPUT indicator is off, no other controller is connected, and `CONFIRM_FRONT_4WIRE_100_OHM_AND_OUTPUT_OFF` has been deliberately set to `True`.

In [ ]:
if not CONFIRM_FRONT_4WIRE_100_OHM_AND_OUTPUT_OFF:
    raise RuntimeError("STOP: reconnect and verify the physical setup before enabling this cell.")

buffer_name = f"oe_m3_{uuid4().hex[:12]}"
config = VisaTransportConfig(
    resource_name=RESOURCE,
    backend="@py",
    required_language="SCPI",
    io_timeout_ms=5_000,
)
result = run_buffer_and_exception_test(PyVisaKeithley2460Transport(config), buffer_name=buffer_name)

audit_transport = PyVisaKeithley2460Transport(config)
try:
    audit_transport.connect()
    independent_audit = {
        "output_state_raw": audit_transport.query(":OUTPut:STATe?"),
        "terminals_raw": audit_transport.query(":ROUTe:TERMinals?"),
        "programmed_current_a_raw": audit_transport.query(":SOURce:CURRent?"),
        "instrument_error_count_raw": audit_transport.query(":SYSTem:ERRor:COUNt?"),
    }
finally:
    audit_transport.close()
require_token(independent_audit["output_state_raw"], ("0", "OFF"), "audit output")
if finite_number(independent_audit["programmed_current_a_raw"], "audit current") != 0:
    raise RuntimeError("independent audit did not find 0 A")
if int(independent_audit["instrument_error_count_raw"]) != result["preexisting_error_count"]:
    raise RuntimeError("independent audit found a changed instrument error count")
result["independent_query_only_audit"] = independent_audit
print(json.dumps(result, indent=2))

## Observed result on 2026-10-05

The connected test completed on instrument serial `04686198`, firmware `1.7.16a`:

- All 20 requested records were stored and retrieved from the unique temporary buffer.
- Mean source readback: 0.999998004 mA; mean four-wire voltage: 102.489834 mV; mean V/I: 102.490038 Ω.
- Relative timestamps spanned 1.568821 s, with mean interval 82.5695 ms (12.1110 Hz).
- Every measurement-status value identified the front terminals; no questionable-measurement bit was set.
- The intentional host exception was observed and cleanup confirmed output OFF and programmed current 0 A.
- The temporary buffer was deleted without increasing the instrument error count.
- A new session independently confirmed output OFF, front terminals, 0 A, and the unchanged two historical LAN errors.

The complete record is `docs/evidence/k2460-2026-10-05-buffer-cleanup.json`. This validates normal in-process exception cleanup only, not process death, network loss, or an instrument-local watchdog.